# Notebook 02 - Domain + Modules + Core 最小实战

这一节从“能跑通一个最小例子”开始，让你看到：
- `domain` 如何承载统一数据结构
- `modules` 如何给出出流策略
- `core` 如何推进仿真并生成时序结果


## 1) 准备：构造最小水库与初始状态

你将看到 `ReservoirSpec`（静态）和 `ReservoirState`（动态）是如何配套的。


In [1]:
from datetime import datetime, timedelta

from pyresops.domain.reservoir import ReservoirSpec, ReservoirState, LevelStorageCurve, DischargeCapacity
from pyresops.domain.program import DispatchProgram, TimeHorizon, ModuleInstance
from pyresops.domain.forecast import ForecastBundle, ForecastSeries

spec = ReservoirSpec(
    id='nb2_res',
    name='Notebook 2 Reservoir',
    dead_level=150.0,
    normal_level=175.0,
    flood_limit_level=155.0,
    design_flood_level=178.0,
    check_flood_level=182.0,
    total_capacity=39.3,
    flood_capacity=22.15,
    level_storage_curve=LevelStorageCurve(
        levels=[135, 145, 155, 165, 175, 185],
        storages=[0, 10, 20, 30, 39.3, 51.6],
    ),
    discharge_capacity=DischargeCapacity(
        levels=[135, 145, 155, 165, 175, 185],
        max_discharges=[0, 5000, 10000, 15000, 20000, 30000],
    ),
)

start = datetime(2024, 7, 1, 0, 0, 0)
state0 = ReservoirState(
    timestamp=start,
    level=160.0,
    storage=25.0,
    inflow=6000.0,
    outflow=6000.0,
)

print('ReservoirSpec ready:', spec.name)
print('Initial state:', state0.model_dump())


ReservoirSpec ready: Notebook 2 Reservoir
Initial state: {'timestamp': datetime.datetime(2024, 7, 1, 0, 0), 'level': 160.0, 'storage': 25.0, 'inflow': 6000.0, 'outflow': 6000.0, 'active_module_id': None, 'metadata': {}}


## 2) 定义最小调度方案与预报

这里先用最简单的 `constant_release` 模块，聚焦理解“仿真引擎如何推进”。


In [2]:
program = DispatchProgram(
    id='nb2_prog',
    name='constant release demo',
    time_horizon=TimeHorizon(start=start, end=start + timedelta(hours=11), time_step=3600),
    module_sequence=[
        ModuleInstance(module_type='constant_release', parameters={'target_flow': 7000.0})
    ],
)

timestamps = [start + timedelta(hours=i) for i in range(12)]
inflow_values = [6000, 6500, 7000, 8000, 9000, 10000, 9500, 8500, 7500, 7000, 6500, 6000]
forecast = ForecastBundle(
    forecast_time=start,
    series=[ForecastSeries(variable='inflow', timestamps=timestamps, values=inflow_values)],
)

print('Program:', program.name)
print('Forecast steps:', len(inflow_values))


Program: constant release demo
Forecast steps: 12


## 3) 跑仿真：`core.engine + modules`

`SimulationService` 内部会：
1. 根据 `module_type` 实例化模块
2. 调用 `SimulationEngine.simulate(...)`
3. 逐步执行水量平衡


In [3]:
from pyresops.services import ProgramService, SimulationService

program_service = ProgramService()
sim_service = SimulationService(spec, program_service.get_module_registry())

result = sim_service.run_simulation(program, state0, forecast)

print('Program ID:', result.program_id)
print('Max level:', round(result.max_level, 3))
print('Min level:', round(result.min_level, 3))
print('Avg outflow:', round(result.avg_outflow, 3))
print('Snapshots:', len(result.snapshots))


Program ID: nb2_prog
Max level: 160.324
Min level: 159.946
Avg outflow: 7000.0
Snapshots: 12


In [4]:
# 看看前几步状态变化
for s in result.snapshots[:5]:
    print(s.timestamp.strftime('%m-%d %H:%M'),
          'level=', round(s.level, 3),
          'in=', round(s.inflow, 1),
          'out=', round(s.outflow, 1),
          'module=', s.active_module)


07-01 00:00 level= 160.0 in= 6000.0 out= 7000.0 module= constant_release
07-01 01:00 level= 159.964 in= 6500.0 out= 7000.0 module= constant_release
07-01 02:00 level= 159.946 in= 7000.0 out= 7000.0 module= constant_release
07-01 03:00 level= 159.946 in= 8000.0 out= 7000.0 module= constant_release
07-01 04:00 level= 159.982 in= 9000.0 out= 7000.0 module= constant_release


## 4) 对比两个模块策略差异

同样的来水，分别用：
- `constant_release`
- `storage_driven`

观察结果差异就能理解 `modules` 层为什么要抽象成可插拔。


In [5]:
from pyresops.services import SimulationService
from pyresops.domain.program import DispatchProgram, ModuleInstance, TimeHorizon

prog_storage = DispatchProgram(
    id='nb2_prog_storage',
    name='storage driven demo',
    time_horizon=TimeHorizon(start=start, end=start + timedelta(hours=11), time_step=3600),
    module_sequence=[
        ModuleInstance(
            module_type='storage_driven',
            parameters={
                'low_storage_threshold': 0.35,
                'high_storage_threshold': 0.75,
                'base_flow': 5000.0,
                'extra_release_rate': 0.2,
            },
        )
    ],
)

res_const = sim_service.run_simulation(program, state0, forecast)
res_store = sim_service.run_simulation(prog_storage, state0, forecast)

print('constant_release avg outflow:', round(res_const.avg_outflow, 2), 'max_level:', round(res_const.max_level, 3))
print('storage_driven  avg outflow:', round(res_store.avg_outflow, 2), 'max_level:', round(res_store.max_level, 3))


constant_release avg outflow: 7000.0 max_level: 160.324
storage_driven  avg outflow: 6901.44 max_level: 160.303


## 5) 小结

你已经看到：
- `domain` 定义了“说话方式”（数据对象）
- `modules` 定义了“如何计算出流”
- `core` 负责把这些规则推进成完整时序轨迹

下一节 `notebook_03_constraints_rules_metrics.ipynb` 会讲：
- 约束怎么校核
- 规则怎么触发动作
- 指标怎么评分
